# LOSO (leave-one-subject-out) on Google Colab — SSVEP

**What this notebook does:** for every subject, train on all the *other* subjects and test on that one, which has never been seen. It is the "new user, no calibration" test the professor asked for (*con todos menos con uno*).

**How Colab works, in 4 lines**
1. This notebook runs on a temporary Google machine (the *runtime*), not on your computer.
2. That machine is **wiped** whenever it disconnects or you close it. Anything not saved to Google Drive is lost.
3. So: code comes from **GitHub** (re-cloned every session), data and results live in **Google Drive** (permanent).
4. You run cells top to bottom with **Shift+Enter**. A `!` at the start of a line runs a terminal command.

**Before the first run (one time only)**
- Add `scripts/benchmarks/benchmark_loso.py` to your repo and push it to GitHub.
- Upload your recordings folder to Google Drive, anywhere you like. It needs one folder per subject named `S01`, `S02`, … with `OO.ebr` inside. Other files in those folders (videos, photos, other sessions) are ignored. The recordings stay private in your Drive and never go to GitHub.
- Menu **Runtime → Change runtime type → T4 GPU** (any GPU works). Then **Save**.

## 1. Check that you got a GPU
You should see a table with a GPU name (e.g. Tesla T4). If you see *command not found*, go back to Runtime → Change runtime type.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive
A pop-up asks you to allow access to your Drive. Accept with the same Google account where you uploaded the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Settings
The only cell you should need to edit.
- `RAW_DIR`: your recordings folder in Drive. To get the exact path: open the folder icon in the left panel, find the folder under `drive/MyDrive/…`, right-click it and choose **Copy path**.
- `BRANCH`: the git branch to run (it must contain `benchmark_loso.py`).
- `LOSO_OUT`: where the LOSO results go. Finished folds are reused on the next run, so use a **new name for every configuration** (other window, channels, epochs, quick test…) or old results get mixed in.

The cell lists every subject folder and says `OK` or `MISSING OO.ebr`.

In [ ]:
RAW_DIR   = '/content/drive/MyDrive/PUT/YOUR/RECORDINGS/FOLDER/HERE'   # folder with S01, S02, ... inside
DRIVE_DIR = '/content/drive/MyDrive/ssvep-eeg-xai'   # results folder in Drive (created automatically)
REPO_URL  = 'https://github.com/Upland1/ssvep-eeg-xai.git'
BRANCH    = 'main'
REPO      = '/content/ssvep-eeg-xai'                  # where the code goes on the Colab machine
LOSO_OUT  = 'reports/loso_w1'                         # one folder per configuration

import os, re
assert os.path.isdir(RAW_DIR), f'Folder not found: {RAW_DIR} -- check RAW_DIR (use Copy path)'
subjects = sorted(d for d in os.listdir(RAW_DIR) if re.fullmatch(r'S\d+', d))
assert subjects, f'No S01, S02, ... folders inside {RAW_DIR}'
for s in subjects:
    print(s, 'OK' if os.path.isfile(os.path.join(RAW_DIR, s, 'OO.ebr')) else 'MISSING OO.ebr')

## 4. Get the code from GitHub
A fresh copy every session, so you always run what is pushed to `BRANCH`. If you change code on your laptop: push, then rerun this cell.

In [ ]:
!rm -rf {REPO}
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1
!ls scripts/benchmarks/benchmark_loso.py

## 5. Install packages
Colab already has numpy, scipy, scikit-learn, pandas and PyTorch with GPU support. **Do not** `pip install -r requirements.txt`: it can replace Colab's GPU PyTorch with a CPU one.

In [ ]:
!pip install -q pyyaml joblib
import torch, sklearn, numpy
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '| sklearn', sklearn.__version__, '| numpy', numpy.__version__)

## 6. Link Drive folders into the repo
`data/raw` points to your recordings folder (read only, nothing is copied or changed), and `reports/` + `outputs/models/` are written straight to Drive, so results survive a disconnect. The scripts don't know the difference.

In [ ]:
import os, shutil
os.chdir(REPO)
links = {'data/raw': RAW_DIR, 'reports': f'{DRIVE_DIR}/reports', 'outputs/models': f'{DRIVE_DIR}/outputs/models'}
for local, target in links.items():
    os.makedirs(target, exist_ok=True)
    os.makedirs(os.path.dirname(local) or '.', exist_ok=True)
    if os.path.islink(local) or os.path.isfile(local):
        os.remove(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(target, local)
    print(f'{local} -> {target}')

## 7. Build the windows from the raw `.ebr` files
Same command you run locally: filtering, quality gate, 1 s windows every 0.25 s, FBCCA/FBCSP filter banks. Takes seconds, so it's simply rebuilt every session (on the Colab disk, not Drive).
Check each subject's line: channels kept and windows rejected.

In [ ]:
!python scripts/build_windows.py --window-sec 1 --stride-sec 0.25 --banks fbcca fbcsp

## 8. Within-subject reference (same data)
LOSO numbers only mean something next to the within-subject numbers **on the same data**. This is your existing window-sweep benchmark: trial-grouped 5-fold CV inside each subject.

In [ ]:
!python scripts/benchmarks/benchmark_window_sweep.py --windows 1 --methods fbcca lda --cv-repeats 5 --out-dir reports/window_sweep_ref

## 9. LOSO — classical methods (about 1 minute)
- `fbcca`: no training at all, the floor every trained method must beat.
- `lda`: FBCSP + FBCCA → Shrinkage-LDA trained on the other subjects.
- `lda_subjnorm`: same, but each subject's features are z-scored with its own statistics (no labels of the test subject are used).

Every finished fold is saved in `LOSO_OUT/folds/`. Re-running a command skips what is done.

In [ ]:
!python scripts/benchmarks/benchmark_loso.py --methods fbcca lda lda_subjnorm --out-dir {LOSO_OUT} 2>&1 | tee -a {LOSO_OUT}_log.txt

## 10. LOSO — neural networks on GPU (the long part)
Same networks and training as `benchmark_nn_sweep.py` (augmentation on, early stopping on 20% of the training trials). One fold = train on 4 subjects, test on the 5th.

**If Colab disconnects:** run cells 2 → 7 again, then this cell. Finished folds are skipped, only the fold in progress is redone.
**Keep this browser tab open** unless your plan includes background execution (Colab Pro+).

In [ ]:
!python scripts/benchmarks/benchmark_loso.py --methods ccnn compact eegnet --out-dir {LOSO_OUT} 2>&1 | tee -a {LOSO_OUT}_log.txt

## 11. Look at the results
`summary.csv` = one row per held-out subject and method. All files are also in your Drive under `ssvep-eeg-xai/reports/` (the LOSO ones in the `LOSO_OUT` folder).

In [ ]:
import pandas as pd
loso = pd.read_csv(f'{LOSO_OUT}/summary.csv')
within = pd.read_csv('reports/window_sweep_ref/summary.csv')
print('LOSO accuracy (%) per held-out subject')
display(loso.pivot(index='Subject', columns='Method', values='Acc (%)').round(1))
table = pd.concat([
    within.groupby('Method')['Acc (%)'].agg(['mean', 'std']).rename(index=lambda m: f'within-subject {m}'),
    loso.groupby('Method')['Acc (%)'].agg(['mean', 'std']).rename(index=lambda m: f'LOSO {m}'),
]).round(2)
display(table)

## 12. Stop the machine when you're done
A connected runtime keeps spending compute units even when idle. Your results are already in Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()